### Simple Agent Application
**Goal**

- Create an agent which can directly answer and also use a calculator tool when required

**Tech stack**
- LangChain
- Ollama (`qwen2.5:1.5b`)

### Import libraries

In [56]:
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

### Get the model

In [57]:
llm = OllamaLLM(
    model="qwen2.5:1.5b",
    temperature=0.0,
    num_predict=100,
    num_ctx=1024,
)

### Tool - Calculator

In [58]:
def calculator(expression: str) -> str:
    try:
        expr = expression.strip().replace("^", "**")
        print("expr: ", expr)
        result = eval(expr, {"__builtins__": {}})
        print("result: ", result)
        return f"{result}"
    except Exception as e:
        return f"Error: {e}"

### Prompt

In [59]:
prompt = PromptTemplate.from_template("""
You are a helpful assistant with one available tool:

TOOL: calculator
USE IT WHEN: question involves math or numbers
ANSWER FORMAT: ACTION: calculator | INPUT: <math expression>

If no tool is needed, answer directly with:
ANSWER: <your answer>

Examples:
Q: What is 10 * 5?  -> ACTION: calculator | INPUT: 10 * 5
Q: Capital of France? -> ANSWER: Paris

Only output one line. No explanations.

Question: {question}
Your reply:
""")

print(prompt)

input_variables=['question'] input_types={} partial_variables={} template='\nYou are a helpful assistant with one available tool:\n\nTOOL: calculator\nUSE IT WHEN: question involves math or numbers\nANSWER FORMAT: ACTION: calculator | INPUT: <math expression>\n\nIf no tool is needed, answer directly with:\nANSWER: <your answer>\n\nExamples:\nQ: What is 10 * 5?  -> ACTION: calculator | INPUT: 10 * 5\nQ: Capital of France? -> ANSWER: Paris\n\nOnly output one line. No explanations.\n\nQuestion: {question}\nYour reply:\n'


In [61]:
def run_agent(question: str) -> str:
    print(f"\n{'='*50}")
    print(f"Question: {question}")
    print(f"{'='*50}")

    prompt_text = prompt.format(question=question)
    
    decision = llm.invoke(prompt_text).strip()
    print(f"LLM decision: {decision}")

    decision_upper = decision.upper()
    print()

    if "ACTION:" in decision_upper:
        start       = decision_upper.index("ACTION:")
        action_part = decision[start:]

        try:
            # Split on "|" to separate tool name from input
            left, right = action_part.split("|", 1)
            tool_name   = left.replace("ACTION:", "").strip().lower()
            tool_input  = right.replace("INPUT:", "").strip()
        except ValueError:
            # "|" missing — fallback: use full question as input
            tool_name  = "calculator"
            tool_input = question

        print(f"Tool: {tool_name}")
        print(f"Tool input: {tool_input}")

        tool_result = calculator(tool_input)
        print(f"Tool result: {tool_result}")

        final_prompt = (
            f"The user asked: {question}\n"
            f"The calculator returned: {tool_result}\n"
            f"Give a short one-sentence answer:"
        )
        # answer = llm.invoke(final_prompt).strip()
        answer = f"The answer to '{question.strip('?')}' is {tool_result}."

        if tool_result not in answer:
            answer = f"The answer is {tool_result}."
    else:
        if "ANSWER:" in decision_upper:
            start  = decision_upper.index("ANSWER:")
            answer = decision[start:].replace("ANSWER:", "").strip()
        else:
            answer = decision
            
    print(f"\nFinal Answer: {answer}")
    return answer

In [62]:
run_agent("What is 25 * 6 ?")


Question: What is 25 * 6 ?
LLM decision: ACTION: calculator | INPUT: 25 * 6

Tool: calculator
Tool input: 25 * 6
expr:  25 * 6
result:  150
Tool result: 150

Final Answer: The answer to 'What is 25 * 6 ' is 150.


"The answer to 'What is 25 * 6 ' is 150."

In [63]:
run_agent("What is the capital of India ?")


Question: What is the capital of India ?
LLM decision: ANSWER: New Delhi


Final Answer: New Delhi


'New Delhi'